# Treino YOLO11 - Deteccao de EPI (Colab) - v5 (+ dataset novo com luvas corrigidas)

Pensado para rodar **sozinho a noite toda**: prepara os dados, treina, e no final compara v4 x v5 no conjunto de teste real (frames do video que nunca entram no treino), salvando tudo no Drive.

**O que muda da v4:** entra o dataset novo unificado (`novos_v5.zip`, 5 datasets baixados), com as luvas e oculos que estavam sem caixa pre-anotados pelo Grounding DINO e revisao parcial no Roboflow. Base: fine-tuning a partir do `best.pt` da v4.

**Antes de rodar:**
1. Suba `novos_v5.zip` para `MyDrive/TCC/yolo_dataset/novos_v5.zip`. Os outros zips (`yolo_v2`, `novos_frames_dataset`, `construction_ppe_dataset`) ja estao la.
2. `Ambiente de execucao > Alterar o tipo` -> **GPU T4**.
3. `Executar tudo`. Deixe a aba aberta e o PC sem hibernar.

**Se a sessao cair** (cota de GPU, desconexao): de manha, `Executar tudo` de novo. O treino **continua de onde parou** (o `last.pt` fica no Drive); se ja tiver terminado, ele pula direto para a avaliacao.

**Resultados** (tudo no Drive, em `MyDrive/TCC/runs/train-colab-v5/`):
- `weights/best.pt` - o modelo v5
- `comparacao_real_world_test.csv` e `.txt` - v4 x v5 no video real, por classe
- `results.csv`, graficos etc. - curvas do treino

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE = '/content/drive/MyDrive/TCC'
ZIPS = f'{DRIVE}/yolo_dataset'
RUNS = f'{DRIVE}/runs'
RUN_NAME = 'train-colab-v5'
RUN_DIR = f'{RUNS}/{RUN_NAME}'
BASE_MODEL = f'{RUNS}/train-colab-v4/weights/best.pt'   # fine-tuning a partir da v4
V4_MODEL = BASE_MODEL
DATA_DIR = '/content'
CLASSES = ['helmet', 'vest', 'gloves', 'boots', 'goggles']

for p in [f'{ZIPS}/yolo_v2.zip', f'{ZIPS}/novos_frames_dataset.zip',
          f'{ZIPS}/construction_ppe_dataset.zip', f'{ZIPS}/novos_v5.zip', BASE_MODEL]:
    assert os.path.exists(p), f'Nao encontrado: {p}'
print('OK - todos os arquivos no Drive')

## Dados: v2 + frames do video + Construction-PPE + dataset novo

In [ ]:
import shutil

!unzip -q -o "{ZIPS}/yolo_v2.zip" -d {DATA_DIR}
!chmod -R u+rwX {DATA_DIR}/train {DATA_DIR}/val {DATA_DIR}/test


def copiar(src_root, pares):
    # pares: [(split de origem, split de destino)]
    for src, dst in pares:
        for tipo in ['images', 'labels']:
            origem = f'{src_root}/{src}/{tipo}'
            for fname in os.listdir(origem):
                shutil.copy(os.path.join(origem, fname), f'{DATA_DIR}/{dst}/{tipo}/{fname}')


# frames reais do video: train_extra (ja com oversample 8x) entra no treino;
# real_world_test fica separado e nunca e treinado
!unzip -q -o "{ZIPS}/novos_frames_dataset.zip" -d /content/novos
copiar('/content/novos/novos_frames_dataset', [('train_extra', 'train')])
REAL_TEST_DIR = '/content/real_world_test'
shutil.copytree('/content/novos/novos_frames_dataset/real_world_test', REAL_TEST_DIR, dirs_exist_ok=True)

!unzip -q -o "{ZIPS}/construction_ppe_dataset.zip" -d /content/cppe
copiar('/content/cppe/construction_ppe_raw', [('train', 'train'), ('val', 'val'), ('test', 'test')])

# dataset novo: no export do Roboflow a validacao se chama 'valid'
!unzip -q -o "{ZIPS}/novos_v5.zip" -d /content/nv5
copiar('/content/nv5/novos_v5', [('train', 'train'), ('valid', 'val'), ('test', 'test')])

In [ ]:
# Checagem antes de gastar GPU
from collections import Counter

for split in ['train', 'val', 'test']:
    imgs = os.listdir(f'{DATA_DIR}/{split}/images')
    lbls = os.listdir(f'{DATA_DIR}/{split}/labels')
    assert imgs and len(imgs) == len(lbls), f'{split}: {len(imgs)} imagens x {len(lbls)} labels'
    caixas = Counter()
    for l in lbls:
        for linha in open(f'{DATA_DIR}/{split}/labels/{l}'):
            if linha.split():
                caixas[CLASSES[int(linha.split()[0])]] += 1
    print(f'{split}: {len(imgs)} imagens | caixas: {dict(caixas)}')
print('real_world_test:', len(os.listdir(f'{REAL_TEST_DIR}/images')), 'imagens (fora do treino)')

with open(f'{DATA_DIR}/data.yaml', 'w') as f:
    f.write(f'train: {DATA_DIR}/train/images\nval: {DATA_DIR}/val/images\ntest: {DATA_DIR}/test/images\n')
    f.write(f'nc: 5\nnames: {CLASSES}\n')
with open(f'{REAL_TEST_DIR}/data.yaml', 'w') as f:
    f.write(f'train: {REAL_TEST_DIR}/images\nval: {REAL_TEST_DIR}/images\n')
    f.write(f'nc: 5\nnames: {CLASSES}\n')

## Treino (retoma sozinho se a sessao tiver caido)

Resultados vao direto pro Drive (`project=`), entao nada se perde se a sessao cair.

In [ ]:
%pip install -q ultralytics
import torch
from ultralytics import YOLO

last = f'{RUN_DIR}/weights/last.pt'
terminado = False
if os.path.exists(last):
    ck = torch.load(last, map_location='cpu', weights_only=False)
    # ao terminar, o Ultralytics remove o otimizador e marca epoch = -1
    terminado = ck.get('epoch', -1) == -1
    print('treino anterior encontrado:', 'TERMINADO' if terminado else f"parou na epoca {ck.get('epoch')}")

if terminado:
    print('Pulando o treino - indo direto para a avaliacao.')
elif os.path.exists(last):
    YOLO(last).train(resume=True)
else:
    YOLO(BASE_MODEL).train(
        data=f'{DATA_DIR}/data.yaml',
        epochs=40,
        patience=15,
        batch=16,
        imgsz=640,
        project=RUNS,
        name=RUN_NAME,
        exist_ok=True,
    )

## Veredito: v4 x v5 no conjunto de teste real

25 frames do video real, anotados a mao, que nunca entraram em nenhum treino. E essa comparacao que vale para o TCC (o split de teste vem da mesma distribuicao do treino e da numero otimista).

In [ ]:
import pandas as pd


def avaliar(modelo, data, nome, split='val'):
    r = YOLO(modelo).val(data=data, imgsz=640, split=split, batch=8, plots=False, verbose=False,
                         project='/content/val_tmp', name=nome, exist_ok=True)
    linhas = [{'modelo': nome, 'classe': 'GERAL', 'mAP50': r.box.map50, 'mAP50-95': r.box.map,
               'precisao': r.box.mp, 'recall': r.box.mr}]
    for i, c in enumerate(r.box.ap_class_index):
        linhas.append({'modelo': nome, 'classe': CLASSES[int(c)], 'mAP50': r.box.ap50[i],
                       'mAP50-95': r.box.ap[i], 'precisao': r.box.p[i], 'recall': r.box.r[i]})
    return linhas


V5_MODEL = f'{RUN_DIR}/weights/best.pt'
res = pd.DataFrame(
    avaliar(V4_MODEL, f'{REAL_TEST_DIR}/data.yaml', 'v4') +
    avaliar(V5_MODEL, f'{REAL_TEST_DIR}/data.yaml', 'v5')
).round(3)
tabela = res.pivot(index='classe', columns='modelo', values=['mAP50', 'recall', 'precisao'])
print('=== CONJUNTO DE TESTE REAL (video) ===')
print(tabela.to_string())

teste = pd.DataFrame(avaliar(V5_MODEL, f'{DATA_DIR}/data.yaml', 'v5-split-teste', split='test')).round(3)
print('\n=== v5 no split de teste (otimista, so para referencia) ===')
print(teste.to_string(index=False))

res.to_csv(f'{RUN_DIR}/comparacao_real_world_test.csv', index=False)
with open(f'{RUN_DIR}/comparacao_real_world_test.txt', 'w') as f:
    f.write('=== CONJUNTO DE TESTE REAL (video) ===\n' + tabela.to_string() + '\n\n')
    f.write('=== v5 no split de teste ===\n' + teste.to_string(index=False) + '\n')
print(f'\nSalvo em {RUN_DIR}/comparacao_real_world_test.csv e .txt')